# StreamSphere Analytics — Data Analyst Portfolio Project

## Project Overview
End-to-end analysis of customer acquisition, subscriptions, revenue, marketing performance, content engagement, and customer value for a fictional streaming platform.

### Tools
- Python
- Pandas
- Matplotlib
- Seaborn
- SQL Server
- Jupyter Notebook


## Business Objectives
- Understand customer distribution and acquisition channels.
- Analyze subscription plans and churn.
- Measure revenue performance.
- Evaluate content and user engagement.
- Measure marketing campaign effectiveness.
- Identify high-value customers.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyodbc

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

## 2. Connect to SQL Server

In [ ]:
server = r'KARAN\SQLEXPRESS'
database = 'StreamSphereAnalytics'

connection = pyodbc.connect(
    f'DRIVER={{ODBC Driver 17 for SQL Server}};'
    f'SERVER={server};'
    f'DATABASE={database};'
    'Trusted_Connection=yes;'
)

print('Connected to SQL Server successfully!')

## 3. Load Data

In [ ]:
customers = pd.read_sql('SELECT * FROM dbo.customers', connection)
plans = pd.read_sql('SELECT * FROM dbo.subscription_plans', connection)
campaigns = pd.read_sql('SELECT * FROM dbo.marketing_campaigns', connection)
subscriptions = pd.read_sql('SELECT * FROM dbo.subscriptions', connection)
content = pd.read_sql('SELECT * FROM dbo.content', connection)
transactions = pd.read_sql('SELECT * FROM dbo.transactions', connection)
events = pd.read_sql('SELECT * FROM dbo.user_events', connection)
geography = pd.read_sql('SELECT * FROM dbo.geography', connection)

tables = {
    'geography': geography,
    'subscription_plans': plans,
    'marketing_campaigns': campaigns,
    'customers': customers,
    'subscriptions': subscriptions,
    'content': content,
    'transactions': transactions,
    'user_events': events
}

for name, df in tables.items():
    print(f'{name}: {df.shape[0]:,} rows | {df.shape[1]} columns')

## 4. Data Type Preparation

In [ ]:
customers['signup_date'] = pd.to_datetime(customers['signup_date'])
subscriptions['start_date'] = pd.to_datetime(subscriptions['start_date'])
subscriptions['end_date'] = pd.to_datetime(subscriptions['end_date'])
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'])

print('Date columns converted successfully.')

## 5. Data Quality & Validation

In [ ]:
print('========== MISSING VALUES ==========')
for name, df in tables.items():
    print(f'\n{name}')
    print(df.isnull().sum())

print('\n========== DUPLICATE KEY CHECKS ==========')
print('Customers:', customers['customer_id'].duplicated().sum())
print('Subscriptions:', subscriptions['subscription_id'].duplicated().sum())
print('Transactions:', transactions['transaction_id'].duplicated().sum())
print('Content:', content['content_id'].duplicated().sum())
print('Events:', events['event_id'].duplicated().sum())

transaction_check = transactions.merge(
    subscriptions[['subscription_id', 'start_date']],
    on='subscription_id', how='left'
)

print('\nTransactions before subscription start:',
      (transaction_check['transaction_date'] < transaction_check['start_date']).sum())

## 6. Customer Analysis

In [ ]:
print('Customers by country:')
display(customers['country'].value_counts())

print('Acquisition channels:')
display(customers['acquisition_channel'].value_counts())

print('Customer age summary:')
display(customers['age'].describe())

print('Customer acquisition by country and channel:')
display(pd.crosstab(customers['country'], customers['acquisition_channel']))

In [ ]:
country_counts = customers['country'].value_counts().sort_values()
plt.figure(figsize=(9,5))
country_counts.plot(kind='barh')
plt.title('Customer Distribution by Country')
plt.xlabel('Customers')
plt.ylabel('Country')
plt.show()

channel_counts = customers['acquisition_channel'].value_counts().sort_values()
plt.figure(figsize=(9,5))
channel_counts.plot(kind='barh')
plt.title('Customer Acquisition by Channel')
plt.xlabel('Customers')
plt.ylabel('Channel')
plt.show()

## 7. Subscription Analysis

In [ ]:
status_counts = subscriptions['subscription_status'].value_counts()
print('Subscription status:')
display(status_counts)

print('Subscriptions by plan:')
display(subscriptions['plan_id'].value_counts())

print('Cancellation reasons:')
display(subscriptions['cancellation_reason'].value_counts())

cancel_rate = (
    subscriptions['subscription_status'].eq('Cancelled').mean() * 100
)
print(f'Overall cancellation rate: {cancel_rate:.2f}%')

In [ ]:
plan_status = pd.crosstab(
    subscriptions['plan_id'],
    subscriptions['subscription_status']
)
display(plan_status)

plan_cancel_rate = (
    subscriptions.assign(cancelled=subscriptions['subscription_status'].eq('Cancelled'))
    .groupby('plan_id')['cancelled']
    .mean().mul(100).sort_values(ascending=False)
)
display(plan_cancel_rate.rename('Cancellation Rate (%)'))

## 8. Revenue Analysis

In [ ]:
successful = transactions[transactions['transaction_status'] == 'Successful'].copy()

total_revenue = successful['amount'].sum()
avg_transaction = successful['amount'].mean()

print(f'Total Successful Revenue: ${total_revenue:,.2f}')
print(f'Average Successful Transaction Value: ${avg_transaction:,.2f}')

revenue_by_plan = (
    successful.merge(
        subscriptions[['subscription_id', 'plan_id']],
        on='subscription_id', how='left'
    ).merge(
        plans[['plan_id', 'plan_name']],
        on='plan_id', how='left'
    ).groupby('plan_name')['amount'].sum()
    .sort_values(ascending=False)
)
print('\nRevenue by plan:')
display(revenue_by_plan)

In [ ]:
revenue_by_country = (
    successful.merge(
        customers[['customer_id', 'country']],
        on='customer_id', how='left'
    ).groupby('country')['amount'].sum()
    .sort_values(ascending=False)
)

display(revenue_by_country)

plt.figure(figsize=(9,5))
revenue_by_country.sort_values().plot(kind='barh')
plt.title('Revenue by Country')
plt.xlabel('Revenue ($)')
plt.ylabel('Country')
plt.show()


In [ ]:
monthly_revenue = (
    successful.assign(month=successful['transaction_date'].dt.to_period('M'))
    .groupby('month')['amount'].sum()
)

plt.figure(figsize=(12,5))
monthly_revenue.plot()
plt.title('Monthly Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Revenue ($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 9. Content & Engagement Analysis

In [ ]:
print('Event type distribution:')
display(events['event_type'].value_counts())

print('Device distribution:')
display(events['device_type'].value_counts())

print('Average watch duration by event type:')
display(events.groupby('event_type')['watch_duration_minutes'].agg(['count','mean','min','max']))

print('Content engagement summary:')
content_engagement = (
    events.groupby('content_id')
    .agg(events=('event_id','count'),
         avg_watch_duration=('watch_duration_minutes','mean'))
    .sort_values('events', ascending=False)
)
display(content_engagement.head(10))

In [ ]:
event_counts = events['event_type'].value_counts()
plt.figure(figsize=(8,5))
event_counts.plot(kind='bar')
plt.title('User Event Distribution')
plt.xlabel('Event Type')
plt.ylabel('Events')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

engagement_by_country = (
    events.merge(
        customers[['customer_id','country']],
        on='customer_id', how='left'
    ).groupby('country')['watch_duration_minutes'].sum()
    .sort_values(ascending=False)
)

plt.figure(figsize=(9,5))
engagement_by_country.sort_values().plot(kind='barh')
plt.title('User Engagement by Country')
plt.xlabel('Total Watch Duration (minutes)')
plt.ylabel('Country')
plt.show()

## 10. Marketing & Customer Value

In [ ]:
campaigns = campaigns.copy()
campaigns['conversion_rate'] = (
    campaigns['conversions'] / campaigns['clicks'].replace(0, np.nan) * 100
)

print('Marketing campaigns by conversion rate:')
display(campaigns[['campaign_name','channel','impressions','clicks','conversions','conversion_rate']]
        .sort_values('conversion_rate', ascending=False))

customer_revenue = (
    successful.groupby('customer_id')['amount']
    .sum().rename('total_revenue')
    .sort_values(ascending=False)
)

print('\nTop 10 customers by revenue:')
display(customer_revenue.head(10))

In [ ]:
plt.figure(figsize=(10,5))
top_campaigns = campaigns.sort_values('conversion_rate', ascending=False).head(10)
plt.bar(top_campaigns['campaign_name'], top_campaigns['conversion_rate'])
plt.title('Marketing Campaign Conversion Rates')
plt.xlabel('Campaign')
plt.ylabel('Conversion Rate (%)')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()

plt.figure(figsize=(9,5))
customer_revenue.head(10).sort_values().plot(kind='barh')
plt.title('Top 10 Customers by Revenue')
plt.xlabel('Revenue ($)')
plt.ylabel('Customer ID')
plt.show()

## 11. Executive KPIs

- **Customers:** 50,000
- **Subscriptions:** 60,000
- **Active Subscriptions:** 39,086
- **Cancelled Subscriptions:** 20,914
- **Overall Churn Rate:** 34.86%
- **Successful Revenue:** $2,264,919.51
- **Average Successful Transaction:** $12.05
- **Top Acquisition Channel:** Instagram
- **Top Revenue Plan:** Standard
- **Top Revenue Country:** United States
- **Top Campaign Conversion Rate:** Email Re-engagement — 20.00%
- **Most Common Event:** Play — 225,189 events


## 12. Business Insights

### Customer & Acquisition
- The United States has the largest customer base and highest revenue contribution.
- Instagram is the leading acquisition channel by customer volume.
- Acquisition mix varies across countries and channels.

### Subscription & Churn
- 39,086 subscriptions are active and 20,914 are cancelled.
- Overall churn is 34.86%.
- Basic has the highest cancellation rate at approximately 35.24%.
- Premium has the lowest cancellation rate at approximately 33.90%.
- Pricing is a major cancellation driver.

### Revenue
- Successful transactions generated $2.26M in revenue.
- Standard generates the highest total revenue.
- The United States is the highest-revenue market.

### Marketing & Engagement
- Email Re-engagement has the highest campaign conversion rate at 20%.
- Play is the most common user event.
- Content-level engagement and watch duration can help prioritize future content investment.


## 13. Business Recommendations

1. **Reduce subscription churn:** investigate pricing, engagement, and competitor-related cancellation drivers and create targeted retention offers.
2. **Optimize pricing:** evaluate plan pricing and promotional offers because 'Too Expensive' is a major cancellation reason.
3. **Strengthen acquisition channels:** scale channels based on both customer volume and conversion efficiency rather than volume alone.
4. **Focus on high-revenue markets:** continue retention and acquisition investment in the United States and India.
5. **Improve content strategy:** identify content characteristics associated with high engagement and watch duration.
6. **Prioritize high-value customers:** use revenue and CLV analysis for targeted retention and loyalty strategies.


## Conclusion

The StreamSphere Analytics project demonstrates an end-to-end Data Analyst workflow: extracting data from SQL Server, validating relational data quality, preparing data in Python, performing exploratory analysis, visualizing KPIs, and translating findings into actionable business recommendations.
